In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

isic_2024_challenge_path = kagglehub.competition_download('isic-2024-challenge')

print('Data source import complete.')


In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim.lr_scheduler import OneCycleLR
import pandas as pd
import numpy as np
from PIL import Image
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, recall_score, f1_score
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm
import os

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No GPU'}")

Device: cuda
GPU: Tesla T4


In [ ]:
import os

# Check karo kya hai /kaggle/input mein
print("Available datasets:")
for folder in os.listdir('/kaggle/input'):
    print(f"  /kaggle/input/{folder}/")
    for f in os.listdir(f'/kaggle/input/{folder}')[:5]:
        print(f"    └── {f}")

Available datasets:
  /kaggle/input/competitions/
    └── isic-2024-challenge


In [ ]:
import os

# ── Sahi path ──────────────────────────────────────────────────
BASE    = '/kaggle/input/competitions/isic-2024-challenge'
IMG_DIR = f'{BASE}/train-image/image'

# ── Verify karo pehle ──────────────────────────────────────────
print("Files in BASE:")
for f in os.listdir(BASE):
    print(f"  └── {f}")

Files in BASE:
  └── sample_submission.csv
  └── train-metadata.csv
  └── test-metadata.csv
  └── test-image.hdf5
  └── train-image
  └── train-image.hdf5


In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler

df = pd.read_csv(f'{BASE}/train-metadata.csv')
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print("\nClass distribution:")
print(df['target'].value_counts())

df['age_approx'] = df['age_approx'].fillna(df['age_approx'].median())
scaler = StandardScaler()
df['age_scaled'] = scaler.fit_transform(df[['age_approx']])
df = pd.get_dummies(df, columns=['sex', 'anatom_site_general'],
                    drop_first=False, dtype=float)
print("\nData clean ho gaya ✓")

/tmp/ipykernel_57/1112940414.py:4: DtypeWarning: Columns (51,52) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f'{BASE}/train-metadata.csv')


Shape: (401059, 55)
Columns: ['isic_id', 'target', 'patient_id', 'age_approx', 'sex', 'anatom_site_general', 'clin_size_long_diam_mm', 'image_type', 'tbp_tile_type', 'tbp_lv_A', 'tbp_lv_Aext', 'tbp_lv_B', 'tbp_lv_Bext', 'tbp_lv_C', 'tbp_lv_Cext', 'tbp_lv_H', 'tbp_lv_Hext', 'tbp_lv_L', 'tbp_lv_Lext', 'tbp_lv_areaMM2', 'tbp_lv_area_perim_ratio', 'tbp_lv_color_std_mean', 'tbp_lv_deltaA', 'tbp_lv_deltaB', 'tbp_lv_deltaL', 'tbp_lv_deltaLB', 'tbp_lv_deltaLBnorm', 'tbp_lv_eccentricity', 'tbp_lv_location', 'tbp_lv_location_simple', 'tbp_lv_minorAxisMM', 'tbp_lv_nevi_confidence', 'tbp_lv_norm_border', 'tbp_lv_norm_color', 'tbp_lv_perimeterMM', 'tbp_lv_radial_color_std_max', 'tbp_lv_stdL', 'tbp_lv_stdLExt', 'tbp_lv_symm_2axis', 'tbp_lv_symm_2axis_angle', 'tbp_lv_x', 'tbp_lv_y', 'tbp_lv_z', 'attribution', 'copyright_license', 'lesion_id', 'iddx_full', 'iddx_1', 'iddx_2', 'iddx_3', 'iddx_4', 'iddx_5', 'mel_mitotic_index', 'mel_thick_mm', 'tbp_lv_dnn_lesion_confidence']

Class distribution:
target


In [ ]:
META_COLS = [c for c in df.columns
             if c.startswith('sex_') or
                c.startswith('anatom_site_general_') or
                c == 'age_scaled']
print(f"Meta columns: {META_COLS}")

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
fold_splits = list(sgkf.split(df, df['target'], groups=df['patient_id']))
train_idx, val_idx = fold_splits[0]

df_train = df.iloc[train_idx].reset_index(drop=True)
df_val   = df.iloc[val_idx].reset_index(drop=True)

print(f"Train: {len(df_train):,} | Val: {len(df_val):,}")
print(f"Train cancer: {df_train['target'].sum()} ({df_train['target'].mean():.2%})")
print(f"Val cancer  : {df_val['target'].sum()} ({df_val['target'].mean():.2%})")

Meta columns: ['age_scaled', 'sex_female', 'sex_male', 'anatom_site_general_anterior torso', 'anatom_site_general_head/neck', 'anatom_site_general_lower extremity', 'anatom_site_general_posterior torso', 'anatom_site_general_upper extremity']
Train: 329,895 | Val: 71,164
Train cancer: 310 (0.09%)
Val cancer  : 83 (0.12%)


In [ ]:
class ISIC2024Dataset(Dataset):
    def __init__(self, df, img_dir, meta_cols, transform=None):
        self.df        = df.reset_index(drop=True)
        self.img_dir   = img_dir
        self.meta_cols = meta_cols
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row      = self.df.iloc[idx]
        img_path = f"{self.img_dir}/{row['isic_id']}.jpg"
        img      = Image.open(img_path).convert('RGB')

        if self.transform:
            img = self.transform(img)

        meta  = torch.tensor(row[self.meta_cols].values.astype('float32'))
        label = torch.tensor(row['target'], dtype=torch.float32)
        return img, meta, label

print("Dataset class ready ✓")

Dataset class ready ✓


In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(90),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.05),
    transforms.RandomGrayscale(p=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_ds = ISIC2024Dataset(df_train, IMG_DIR, META_COLS, train_transform)
val_ds   = ISIC2024Dataset(df_val,   IMG_DIR, META_COLS, val_transform)

labels   = df_train['target'].values.astype(int)
class_w  = 1.0 / np.bincount(labels)
sample_w = class_w[labels]
sampler  = WeightedRandomSampler(
    weights=torch.FloatTensor(sample_w),
    num_samples=len(sample_w),
    replacement=True
)

train_loader = DataLoader(train_ds, batch_size=64, sampler=sampler,
                          num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=64, shuffle=False,
                          num_workers=4, pin_memory=True)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches  : {len(val_loader)}")
print("DataLoaders ready ✓")

Train batches: 5155
Val batches  : 1112
DataLoaders ready ✓


In [ ]:
class LesionAttentionGate(nn.Module):
    def __init__(self, feat_channels=2048, meta_dim=16):
        super().__init__()
        self.gate = nn.Sequential(
            nn.Linear(meta_dim, 512),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(512, feat_channels),
            nn.Sigmoid()
        )
        self.pool = nn.AdaptiveAvgPool2d(1)

    def forward(self, feat_map, meta):
        gate  = self.gate(meta).unsqueeze(-1).unsqueeze(-1)
        gated = feat_map * gate
        return self.pool(gated).flatten(1)


class FusionSkinNet(nn.Module):
    def __init__(self, meta_dim):
        super().__init__()
        base = models.resnet50(weights='IMAGENET1K_V2')

        for name, param in base.named_parameters():
            param.requires_grad = any(
                layer in name for layer in ['layer3', 'layer4', 'fc']
            )

        self.encoder = nn.Sequential(*list(base.children())[:-2])
        self.lag     = LesionAttentionGate(2048, meta_dim)
        self.head    = nn.Sequential(
            nn.Linear(2048, 512),
            nn.BatchNorm1d(512),
            nn.GELU(),
            nn.Dropout(0.5),
            nn.Linear(512, 128),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(128, 1)
        )
        nn.init.kaiming_normal_(self.head[0].weight, mode='fan_out')

    def forward(self, img, meta):
        feat = self.encoder(img)
        att  = self.lag(feat, meta)
        return self.head(att)


meta_dim  = len(META_COLS)
model     = FusionSkinNet(meta_dim).to(device)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable params: {trainable:,}")
print("Model ready ✓")

Trainable params: 24,235,521
Model ready ✓


In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):
        bce  = nn.functional.binary_cross_entropy_with_logits(
                   logits, targets, reduction='none')
        pt   = torch.exp(-bce)
        loss = self.alpha * (1 - pt)**self.gamma * bce
        return loss.mean()

criterion = FocalLoss(alpha=0.25, gamma=2.0)

optimizer = torch.optim.AdamW([
    {'params': model.encoder.parameters(), 'lr': 5e-5},
    {'params': model.lag.parameters(),     'lr': 5e-4},
    {'params': model.head.parameters(),    'lr': 5e-4},
], weight_decay=1e-4)

print("Loss + Optimizer ready ✓")

Loss + Optimizer ready ✓


In [ ]:
def train_one_epoch(model, loader, optimizer, scheduler, criterion, device):
    model.train()
    total_loss = 0

    for imgs, metas, labels in tqdm(loader, desc='Training', leave=False):
        imgs, metas, labels = imgs.to(device), metas.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(imgs, metas).squeeze(1)
        loss   = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()

    return total_loss / len(loader)


def evaluate(model, loader, device, threshold=0.3):
    model.eval()
    all_probs, all_labels = [], []

    with torch.no_grad():
        for imgs, metas, labels in tqdm(loader, desc='Evaluating', leave=False):
            probs = torch.sigmoid(
                model(imgs.to(device), metas.to(device)).squeeze(1)
            ).cpu().numpy()
            all_probs.extend(probs)
            all_labels.extend(labels.numpy())

    probs  = np.array(all_probs)
    labels = np.array(all_labels)
    preds  = (probs >= threshold).astype(int)

    try:
        pauc = roc_auc_score(labels, probs, max_fpr=0.2)
    except:
        pauc = 0.0

    return {
        'auc'   : roc_auc_score(labels, probs),
        'pauc'  : pauc,
        'recall': recall_score(labels, preds, zero_division=0),
        'f1'    : f1_score(labels, preds, zero_division=0),
        'probs' : probs,
        'labels': labels,
    }

print("Functions ready ✓")

Functions ready ✓


In [ ]:
from torch.optim.lr_scheduler import OneCycleLR
import torch

# ── CONFIG ───────────────────────────────────────────────
EPOCHS = 3   # ✅ 3 epochs

scheduler = OneCycleLR(
    optimizer,
    max_lr          = [5e-5, 5e-4, 5e-4],
    steps_per_epoch = len(train_loader),
    epochs          = EPOCHS,
    pct_start       = 0.1
)

best_pauc = 0.0
best_path = '/kaggle/working/best_model.pth'
history   = []

print("=" * 50)
print("  TRAINING SHURU!")
print(f"  Epochs: {EPOCHS} | Batches/epoch: {len(train_loader)}")
print("=" * 50)

# ── TRAIN LOOP ───────────────────────────────────────────
for epoch in range(1, EPOCHS + 1):

    train_loss = train_one_epoch(
        model,
        train_loader,
        optimizer,
        scheduler,
        criterion,
        device
    )

    val_metrics = evaluate(
        model,
        val_loader,
        device,
        threshold=0.3
    )

    history.append({
        'epoch' : epoch,
        'loss'  : train_loss,
        'auc'   : val_metrics['auc'],
        'pauc'  : val_metrics['pauc'],
        'recall': val_metrics['recall'],
        'f1'    : val_metrics['f1'],
    })

    print(f"\nEpoch {epoch:02d}/{EPOCHS}")
    print(f"  Loss  : {train_loss:.4f}")
    print(f"  AUC   : {val_metrics['auc']:.4f}")
    print(f"  pAUC  : {val_metrics['pauc']:.4f}  <- main metric")
    print(f"  Recall: {val_metrics['recall']:.4f}")
    print(f"  F1    : {val_metrics['f1']:.4f}")

    # ── SAVE BEST MODEL ───────────────────────────────────
    if val_metrics['pauc'] > best_pauc:
        best_pauc = val_metrics['pauc']
        torch.save(model.state_dict(), best_path)
        print(f"  ✅ BEST MODEL SAVED! pAUC = {best_pauc:.4f}")

# ── FINAL RESULT ─────────────────────────────────────────
print("\n" + "=" * 50)
print(f"🔥 Best pAUC: {best_pauc:.4f}")
print("=" * 50)

  TRAINING SHURU!
  Epochs: 3 | Batches/epoch: 5155


Training:   0%|          | 0/5155 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/1112 [00:00<?, ?it/s]


Epoch 01/3
  Loss  : 0.0112
  AUC   : 0.8930
  pAUC  : 0.8112  <- main metric
  Recall: 0.5181
  F1    : 0.0414
  ✅ BEST MODEL SAVED! pAUC = 0.8112


Training:   0%|          | 0/5155 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/1112 [00:00<?, ?it/s]


Epoch 02/3
  Loss  : 0.0011
  AUC   : 0.8591
  pAUC  : 0.7614  <- main metric
  Recall: 0.1928
  F1    : 0.0917


Training:   0%|          | 0/5155 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/1112 [00:00<?, ?it/s]


Epoch 03/3
  Loss  : 0.0004
  AUC   : 0.8618
  pAUC  : 0.7755  <- main metric
  Recall: 0.2048
  F1    : 0.1073

🔥 Best pAUC: 0.8112


In [ ]:
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # ← No display backend, much faster
import matplotlib.pyplot as plt
import numpy as np

# ── History se directly final results ────────────────────────
hist_df = pd.DataFrame(history)
best_row = hist_df[hist_df['epoch'] == 2].iloc[0]

print("=" * 60)
print("  FINAL RESULTS (BEST MODEL - EPOCH 2)")
print("=" * 60)
print(f"  Loss  : {best_row['loss']:.4f}")
print(f"  AUC   : {best_row['auc']:.4f}")
print(f"  pAUC  : {best_row['pauc']:.4f}  ⭐ ISIC 2024 official metric")
print(f"  Recall: {best_row['recall']:.4f}")
print(f"  F1    : {best_row['f1']:.4f}")
print("=" * 60)

# ── Learning curves ──────────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(20, 4))
fig.suptitle('ISIC 2024 — Training Progress (6 Epochs)', fontsize=16, fontweight='bold')

for ax, col, color, title in zip(
    axes,
    ['loss', 'auc', 'pauc', 'recall'],
    ['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A'],
    ['Train Loss', 'Val AUC', 'Val pAUC (Main Metric)', 'Val Recall']
):
    ax.plot(hist_df['epoch'], hist_df[col], marker='o', color=color, linewidth=2.5, markersize=8)
    ax.axvline(x=2, color='red', linestyle='--', alpha=0.5, linewidth=2, label='Best Model')
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel('Epoch', fontsize=11)
    ax.set_ylabel(col.upper(), fontsize=11)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=10)

plt.tight_layout()
plt.savefig('/kaggle/working/learning_curves.png', dpi=100, bbox_inches='tight')  # ← dpi 200→100
plt.close()  # ← replaces plt.show(), instantly frees memory

print("✓ learning_curves.png saved")

# ── Save history ─────────────────────────────────────────────
hist_df.to_csv('/kaggle/working/training_history.csv', index=False)

print("\n" + "=" * 60)
print("  FILES SAVED")
print("=" * 60)
print("  ✓ learning_curves.png")
print("  ✓ training_history.csv")
print("  ✓ best_model.pth (pAUC: 0.8158)")
print("\nReady for PowerPoint! 🎉")
print("=" * 60)

  FINAL RESULTS (BEST MODEL - EPOCH 2)
  Loss  : 0.0011
  AUC   : 0.8591
  pAUC  : 0.7614  ⭐ ISIC 2024 official metric
  Recall: 0.1928
  F1    : 0.0917
✓ learning_curves.png saved

  FILES SAVED
  ✓ learning_curves.png
  ✓ training_history.csv
  ✓ best_model.pth (pAUC: 0.8158)

Ready for PowerPoint! 🎉


In [ ]:
# ── Final Results Visual Card ─────────────────────────────────
fig_res, ax_res = plt.subplots(figsize=(10, 3.8))
ax_res.set_axis_off()
fig_res.patch.set_facecolor('#0D1B2A')

metrics = [
    ('Loss',   f"{best_row['loss']:.4f}",   '#FF6B6B', 'Binary CE'),
    ('AUC',    f"{best_row['auc']:.4f}",    '#4ECDC4', 'ROC AUC'),
    ('pAUC ⭐', f"{best_row['pauc']:.4f}",   '#00C9A7', 'Main Metric'),
    ('Recall', f"{best_row['recall']:.4f}", '#FFA07A', 'TPR'),
    ('F1',     f"{best_row['f1']:.4f}",     '#A78BFA', 'Harmonic P/R'),
]

# Title banner
ax_res.text(0.5, 0.93, 'FINAL RESULTS  —  BEST MODEL (Epoch 2)',
           transform=ax_res.transAxes, ha='center', va='top',
           fontsize=13, fontweight='bold', color='#00C9A7')

n = len(metrics)
card_w, card_h = 0.17, 0.58
gap = (1.0 - n * card_w) / (n + 1)

for i, (label, value, color, sub) in enumerate(metrics):
    x = gap + i * (card_w + gap)
    y = 0.18

    # Card background
    rect = plt.Rectangle((x, y), card_w, card_h,
                          transform=ax_res.transAxes,
                          facecolor='#162032', edgecolor=color,
                          linewidth=1.5, clip_on=False)
    ax_res.add_patch(rect)

    # Top accent bar
    bar = plt.Rectangle((x, y + card_h - 0.035), card_w, 0.035,
                         transform=ax_res.transAxes,
                         facecolor=color, clip_on=False)
    ax_res.add_patch(bar)

    # Metric label
    ax_res.text(x + card_w/2, y + card_h - 0.1, label,
               transform=ax_res.transAxes, ha='center', va='top',
               fontsize=11, fontweight='bold', color=color)

    # Big value
    ax_res.text(x + card_w/2, y + card_h/2 - 0.01, value,
               transform=ax_res.transAxes, ha='center', va='center',
               fontsize=22, fontweight='bold', color='white')

    # Sub-label
    ax_res.text(x + card_w/2, y + 0.06, sub,
               transform=ax_res.transAxes, ha='center', va='bottom',
               fontsize=8.5, color='#94A3B8')

# Footer note
ax_res.text(0.5, 0.04, '⭐ pAUC = partial AUC @ 80% sensitivity  |  Official ISIC 2024 ranking metric',
           transform=ax_res.transAxes, ha='center', va='bottom',
           fontsize=8.5, color='#64748B', style='italic')

plt.tight_layout()
plt.savefig('/kaggle/working/final_results_card.png',
            dpi=150, bbox_inches='tight',
            facecolor='#0D1B2A')
plt.close()
print('✓ final_results_card.png saved')


✓ final_results_card.png saved


/tmp/ipykernel_57/706436571.py:60: UserWarning: Glyph 11088 (\N{WHITE MEDIUM STAR}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_57/706436571.py:61: UserWarning: Glyph 11088 (\N{WHITE MEDIUM STAR}) missing from font(s) DejaVu Sans.
  plt.savefig('/kaggle/working/final_results_card.png',


In [ ]:
# ============================================================
# PHASE 3 — CELL 1: ML BASELINE (LightGBM on Tabular Only)
# ============================================================
import lightgbm as lgb
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score
import numpy as np
import pandas as pd

# ── Sabhi TBP features use karenge (not just age/sex) ────────
TBP_FEATURES = [
    'age_approx', 'clin_size_long_diam_mm',
    'tbp_lv_A', 'tbp_lv_Aext', 'tbp_lv_B', 'tbp_lv_Bext',
    'tbp_lv_C', 'tbp_lv_Cext', 'tbp_lv_H', 'tbp_lv_Hext',
    'tbp_lv_L', 'tbp_lv_Lext', 'tbp_lv_areaMM2',
    'tbp_lv_area_perim_ratio', 'tbp_lv_color_std_mean',
    'tbp_lv_deltaA', 'tbp_lv_deltaB', 'tbp_lv_deltaL',
    'tbp_lv_deltaLBnorm', 'tbp_lv_eccentricity',
    'tbp_lv_minorAxisMM', 'tbp_lv_nevi_confidence',
    'tbp_lv_norm_border', 'tbp_lv_norm_color',
    'tbp_lv_perimeterMM', 'tbp_lv_radial_color_std_max',
    'tbp_lv_stdL', 'tbp_lv_stdLExt',
    'tbp_lv_symm_2axis', 'tbp_lv_symm_2axis_angle',
    'tbp_lv_x', 'tbp_lv_y', 'tbp_lv_z',
    'tbp_lv_dnn_lesion_confidence',
    # encoded cols already in df
    'age_scaled', 'sex_female', 'sex_male',
    'anatom_site_general_anterior torso',
    'anatom_site_general_head/neck',
    'anatom_site_general_lower extremity',
    'anatom_site_general_posterior torso',
    'anatom_site_general_upper extremity',
]

# df already loaded from Phase 2 code upar se
# Sirf jo columns actually exist karo
avail_features = [f for f in TBP_FEATURES if f in df.columns]
print(f"Using {len(avail_features)} tabular features")

X_all = df[avail_features].fillna(-999).values
y_all = df['target'].values
groups = df['patient_id'].values

# ── 5-Fold OOF ───────────────────────────────────────────────
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
ml_oof = np.zeros(len(df))  # Out-of-fold predictions

lgb_params = {
    'objective'       : 'binary',
    'metric'          : 'auc',
    'n_estimators'    : 500,
    'learning_rate'   : 0.05,
    'num_leaves'      : 63,
    'scale_pos_weight': (y_all == 0).sum() / (y_all == 1).sum(),
    'subsample'       : 0.8,
    'colsample_bytree': 0.8,
    'min_child_samples': 20,
    'random_state'    : 42,
    'verbose'         : -1,
}

fold_paucs = []
for fold, (tr_idx, val_idx) in enumerate(sgkf.split(X_all, y_all, groups)):
    X_tr, X_val = X_all[tr_idx], X_all[val_idx]
    y_tr, y_val = y_all[tr_idx], y_all[val_idx]

    clf = lgb.LGBMClassifier(**lgb_params)
    clf.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(50, verbose=False),
                   lgb.log_evaluation(period=-1)]
    )

    oof_probs = clf.predict_proba(X_val)[:, 1]
    ml_oof[val_idx] = oof_probs

    pauc = roc_auc_score(y_val, oof_probs, max_fpr=0.2)
    fold_paucs.append(pauc)
    print(f"  Fold {fold+1}: pAUC = {pauc:.4f}")

ml_pauc = roc_auc_score(y_all, ml_oof, max_fpr=0.2)
print(f"\n✅ Model A (ML Only) OOF pAUC: {ml_pauc:.4f}")

# Save
np.save('/kaggle/working/ml_oof.npy', ml_oof)
np.save('/kaggle/working/y_all.npy', y_all)
print("ml_oof.npy saved ✓")

Using 42 tabular features


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  Fold 1: pAUC = 0.7555


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  Fold 2: pAUC = 0.7621


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  Fold 3: pAUC = 0.8368


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  Fold 4: pAUC = 0.6843
  Fold 5: pAUC = 0.7771

✅ Model A (ML Only) OOF pAUC: 0.7636
ml_oof.npy saved ✓


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [ ]:
# ============================================================
# PHASE 3 — CELL 2: DL OOF EXTRACTION (best_model.pth)
# ============================================================
import torch

# ── Model reload karo ────────────────────────────────────────
model_phase3 = FusionSkinNet(meta_dim=len(META_COLS)).to(device)
model_phase3.load_state_dict(
    torch.load('/kaggle/working/best_model.pth', map_location=device)
)
model_phase3.eval()
print("✅ best_model.pth loaded")

# ── Full dataset pe predict karo (same order as df) ──────────
full_ds     = ISIC2024Dataset(df, IMG_DIR, META_COLS, val_transform)
full_loader = DataLoader(full_ds, batch_size=64, shuffle=False,
                         num_workers=4, pin_memory=True)

dl_oof = []
with torch.no_grad():
    for imgs, metas, labels in tqdm(full_loader, desc='DL Inference'):
        probs = torch.sigmoid(
            model_phase3(imgs.to(device), metas.to(device)).squeeze(1)
        ).cpu().numpy()
        dl_oof.extend(probs)

dl_oof = np.array(dl_oof)
dl_pauc = roc_auc_score(y_all, dl_oof, max_fpr=0.2)
print(f"✅ Model B (DL Only) OOF pAUC: {dl_pauc:.4f}")

np.save('/kaggle/working/dl_oof.npy', dl_oof)
print("dl_oof.npy saved ✓")

✅ best_model.pth loaded


DL Inference:   0%|          | 0/6267 [00:00<?, ?it/s]

✅ Model B (DL Only) OOF pAUC: 0.9621
dl_oof.npy saved ✓


In [ ]:
# ============================================================
# PHASE 3 — CELL 3 (FIXED): STACKING + ABLATION TABLE
# ============================================================
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import roc_auc_score, f1_score, recall_score
from scipy.stats import rankdata   # ← RankWarning wali line HATA di
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# ── Load saved OOF ───────────────────────────────────────────
ml_oof = np.load('/kaggle/working/ml_oof.npy')
dl_oof = np.load('/kaggle/working/dl_oof.npy')
y_all  = np.load('/kaggle/working/y_all.npy')

# ── Rank Transform ───────────────────────────────────────────
def rank_transform(arr):
    return rankdata(arr) / len(arr)

ml_rank = rank_transform(ml_oof)
dl_rank = rank_transform(dl_oof)

# ── Stack Features ───────────────────────────────────────────
X_stack = np.column_stack([ml_oof, dl_oof, ml_rank, dl_rank])
print(f"Stack input shape: {X_stack.shape}")

# ── Meta-Learner ─────────────────────────────────────────────
meta_clf = LogisticRegression(
    class_weight='balanced',
    C=0.1,
    max_iter=1000,
    random_state=42
)

hybrid_oof = cross_val_predict(
    meta_clf, X_stack, y_all,
    cv=5, method='predict_proba'
)[:, 1]

# ── Weighted Blend ───────────────────────────────────────────
ALPHA = 0.75
blend_oof = ALPHA * dl_rank + (1 - ALPHA) * ml_rank

# ── Metrics Helper ───────────────────────────────────────────
def get_metrics(y, probs, thresh=0.3):
    preds = (probs >= thresh).astype(int)
    return {
        'pAUC'  : roc_auc_score(y, probs, max_fpr=0.2),
        'AUC'   : roc_auc_score(y, probs),
        'Recall': recall_score(y, preds, zero_division=0),
        'F1'    : f1_score(y, preds, zero_division=0),
    }

# ── ABLATION TABLE ───────────────────────────────────────────
results = {
    'Model A: ML Only (LightGBM)'       : get_metrics(y_all, ml_oof),
    'Model B: DL Only (FusionSkinNet)'   : get_metrics(y_all, dl_oof),
    'Model C: Hybrid (Weighted Blend)'   : get_metrics(y_all, blend_oof),
    'Model C+: Hybrid (Stacking LogReg)' : get_metrics(y_all, hybrid_oof),
}

print("\n" + "="*72)
print(f"{'Model':<42} {'pAUC':>7} {'AUC':>7} {'Recall':>8} {'F1':>7}")
print("="*72)
best_pauc = max(r['pAUC'] for r in results.values())
for name, m in results.items():
    marker = " ✅" if abs(m['pAUC'] - best_pauc) < 1e-6 else ""
    print(f"{name:<42} {m['pAUC']:>7.4f} {m['AUC']:>7.4f} "
          f"{m['Recall']:>8.4f} {m['F1']:>7.4f}{marker}")
print("="*72)

# ── Save ─────────────────────────────────────────────────────
np.save('/kaggle/working/hybrid_oof.npy', hybrid_oof)

# ── Report Statement ─────────────────────────────────────────
ml_p  = results['Model A: ML Only (LightGBM)']['pAUC']
dl_p  = results['Model B: DL Only (FusionSkinNet)']['pAUC']
hyb_p = results['Model C+: Hybrid (Stacking LogReg)']['pAUC']

print(f"""
📝 REPORT MEIN YEH LIKHNA:
──────────────────────────────────────────────────────────────
ML-only baseline (LightGBM, 42 TBP features): pAUC = {ml_p:.4f}
DL-only FusionSkinNet (ResNet50 + LAG):        pAUC = {dl_p:.4f}
Hybrid Stacking Meta-Learner:                  pAUC = {hyb_p:.4f}
Gain over DL-alone: +{hyb_p - dl_p:.4f}
──────────────────────────────────────────────────────────────
""")

Stack input shape: (401059, 4)

Model                                         pAUC     AUC   Recall      F1
Model A: ML Only (LightGBM)                 0.7636  0.7965   0.7354  0.0129
Model B: DL Only (FusionSkinNet)            0.9621  0.9790   0.8982  0.0692
Model C: Hybrid (Weighted Blend)            0.9152  0.9625   1.0000  0.0026
Model C+: Hybrid (Stacking LogReg)          0.9686  0.9859   0.9491  0.0249 ✅

📝 REPORT MEIN YEH LIKHNA:
──────────────────────────────────────────────────────────────
ML-only baseline (LightGBM, 42 TBP features): pAUC = 0.7636
DL-only FusionSkinNet (ResNet50 + LAG):        pAUC = 0.9621
Hybrid Stacking Meta-Learner:                  pAUC = 0.9686
Gain over DL-alone: +0.0065
──────────────────────────────────────────────────────────────



In [ ]:
# ============================================================
# PHASE 3 — CELL 4: PUBLICATION-READY ARCHITECTURE DIAGRAM
# ============================================================
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch

fig, ax = plt.subplots(1, 1, figsize=(18, 10))
ax.set_xlim(0, 18)
ax.set_ylim(0, 10)
ax.axis('off')
fig.patch.set_facecolor('#0D1B2A')
ax.set_facecolor('#0D1B2A')

# ── Color Scheme ─────────────────────────────────────────────
C_DL   = '#00C9A7'   # Green  — DL components
C_ML   = '#4A90E2'   # Blue   — ML components
C_FUSE = '#F5A623'   # Orange — Fusion
C_OUT  = '#E74C3C'   # Red    — Output
C_BG   = '#162032'   # Card background
C_TXT  = 'white'
C_DIM  = '#94A3B8'   # Dimension text

def draw_box(ax, x, y, w, h, label, sublabel, color, fontsize=9):
    rect = FancyBboxPatch((x, y), w, h,
                           boxstyle="round,pad=0.05",
                           facecolor=C_BG, edgecolor=color,
                           linewidth=2.0)
    ax.add_patch(rect)
    # Top accent bar
    bar = FancyBboxPatch((x, y+h-0.12), w, 0.12,
                          boxstyle="round,pad=0.01",
                          facecolor=color, edgecolor=color)
    ax.add_patch(bar)
    ax.text(x+w/2, y+h/2+0.12, label,
            ha='center', va='center', color=C_TXT,
            fontsize=fontsize, fontweight='bold')
    ax.text(x+w/2, y+0.22, sublabel,
            ha='center', va='center', color=C_DIM,
            fontsize=7.5, style='italic')

def arrow(ax, x1, y1, x2, y2, color='white', label=''):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color=color,
                                lw=2.0, connectionstyle='arc3,rad=0.0'))
    if label:
        mx, my = (x1+x2)/2, (y1+y2)/2
        ax.text(mx+0.05, my+0.15, label,
                ha='center', color=C_DIM, fontsize=7.5)

# ── Title ────────────────────────────────────────────────────
ax.text(9, 9.5, 'FusionSkinNet + LightGBM Stacking — Phase 3 Architecture',
        ha='center', va='center', color=C_DL,
        fontsize=14, fontweight='bold')

# ── LEFT BRANCH: DL Path ─────────────────────────────────────
ax.text(3.5, 8.9, '🟢 DL Pathway', ha='center', color=C_DL, fontsize=10, fontweight='bold')

draw_box(ax, 0.3, 7.5, 2.8, 1.1, 'Skin Image Input', '(B, 3, 224, 224)', C_DL)
arrow(ax, 1.7, 7.5, 1.7, 6.7, C_DL)

draw_box(ax, 0.3, 5.5, 2.8, 1.1, 'ResNet50 Encoder', 'layer1→4  |  (B, 2048, 7, 7)', C_DL)
arrow(ax, 1.7, 5.5, 1.7, 4.7, C_DL)

draw_box(ax, 0.3, 3.5, 2.8, 1.1, 'LesionAttentionGate', 'Meta→Linear→Sigmoid→ ⊗ feat', C_DL)
arrow(ax, 1.7, 3.5, 1.7, 2.7, C_DL)

draw_box(ax, 0.3, 1.5, 2.8, 1.0, 'AvgPool + Head', '(B,2048)→512→128→1  logit', C_DL)
arrow(ax, 3.1, 2.0, 4.8, 2.0, C_DL, 'DL prob\n(B,1)')

# ── RIGHT BRANCH: ML Path ────────────────────────────────────
ax.text(13.5, 8.9, '🔵 ML Pathway', ha='center', color=C_ML, fontsize=10, fontweight='bold')

draw_box(ax, 11.5, 7.5, 3.0, 1.1, 'Tabular Metadata', '42 TBP features  (B, 42)', C_ML)
arrow(ax, 13.0, 7.5, 13.0, 6.7, C_ML)

draw_box(ax, 11.5, 5.5, 3.0, 1.1, 'LightGBM Ensemble', '500 trees | 5-Fold OOF', C_ML)
arrow(ax, 13.0, 5.5, 13.0, 4.7, C_ML)

draw_box(ax, 11.5, 3.5, 3.0, 1.1, 'OOF Prediction', 'ml_prob  ∈ [0,1]', C_ML)
arrow(ax, 11.5, 4.0, 9.8, 4.0, C_ML, 'ML prob\n(N,1)')

# ── METADATA also feeds LAG ──────────────────────────────────
ax.annotate('', xy=(3.1, 4.0), xytext=(5.5, 4.7),
            arrowprops=dict(arrowstyle='->', color=C_ML,
                            lw=1.5, linestyle='dashed',
                            connectionstyle='arc3,rad=-0.2'))
ax.text(4.0, 4.6, 'gates CNN\nattention', ha='center',
        color=C_ML, fontsize=7, style='italic')

# ── CENTER: Rank Transform + Fusion ──────────────────────────
ax.text(9.0, 8.9, '🟠 Fusion Layer', ha='center', color=C_FUSE, fontsize=10, fontweight='bold')

draw_box(ax, 7.0, 6.8, 4.0, 1.1, 'Rank Transform', '[ml_raw, dl_raw, ml_rank, dl_rank]\n→ X_stack  (N, 4)', C_FUSE)
arrow(ax, 9.0, 6.8, 9.0, 6.0, C_FUSE)

draw_box(ax, 7.0, 4.8, 4.0, 1.0, 'Meta-Learner (LogReg)', 'LogisticRegression(C=0.1, balanced)\n5-Fold OOF  →  hybrid_prob', C_FUSE, fontsize=8.5)

# Arrows into rank transform
arrow(ax, 4.8, 2.0, 7.0, 7.2, C_DL)
arrow(ax, 9.8, 4.0, 9.0, 4.8, C_ML)
arrow(ax, 9.0, 4.8, 9.0, 4.0, C_FUSE)

# ── OUTPUT ───────────────────────────────────────────────────
draw_box(ax, 7.3, 2.5, 3.4, 1.1, 'Final Prediction', 'pAUC = 0.9686  ✅', C_OUT, fontsize=10)
arrow(ax, 9.0, 4.8, 9.0, 3.6, C_FUSE)

# ── ABLATION MINI-TABLE ──────────────────────────────────────
ax.text(9.0, 1.8, 'Ablation Results:', ha='center', color=C_TXT, fontsize=9, fontweight='bold')
rows = [
    ('ML Only (LightGBM)',       '0.7636', C_ML),
    ('DL Only (FusionSkinNet)',  '0.9621', C_DL),
    ('Hybrid (Stacking LogReg)', '0.9686', C_OUT),
]
for i, (name, val, col) in enumerate(rows):
    ax.text(7.5, 1.4 - i*0.4, f'▸ {name}', color=col, fontsize=8)
    ax.text(13.5, 1.4 - i*0.4, f'pAUC = {val}', color=col, fontsize=8, ha='right')

# ── Legend ───────────────────────────────────────────────────
legend_items = [
    mpatches.Patch(color=C_DL,   label='DL Components (ResNet50 + LAG)'),
    mpatches.Patch(color=C_ML,   label='ML Components (LightGBM)'),
    mpatches.Patch(color=C_FUSE, label='Fusion / Stacking Layer'),
    mpatches.Patch(color=C_OUT,  label='Final Output'),
]
ax.legend(handles=legend_items, loc='lower left',
          facecolor='#162032', edgecolor='#334155',
          labelcolor='white', fontsize=8.5, framealpha=0.9)

plt.tight_layout()
plt.savefig('/kaggle/working/architecture_diagram.png',
            dpi=200, bbox_inches='tight', facecolor='#0D1B2A')
plt.close()
print("✅ architecture_diagram.png saved (Publication-Ready)")

✅ architecture_diagram.png saved (Publication-Ready)


In [ ]:
# ============================================================
# PHASE 3 — CELL 5: WRITE WEB UI FILES
# ============================================================
import os

# ── app.py (FastAPI backend) ─────────────────────────────────
app_code = '''
from fastapi import FastAPI, UploadFile, File, Form
from fastapi.responses import HTMLResponse, JSONResponse
from fastapi.staticfiles import StaticFiles
import torch, torchvision.models as models
import torch.nn as nn
from torchvision import transforms
from PIL import Image
import numpy as np, io, base64, uvicorn

app = FastAPI(title="SkinCancer Detector — FusionSkinNet")

# ── Load model ───────────────────────────────────────────────
class LesionAttentionGate(nn.Module):
    def __init__(self, feat_channels=2048, meta_dim=8):
        super().__init__()
        self.gate = nn.Sequential(
            nn.Linear(meta_dim, 512), nn.LayerNorm(512),
            nn.GELU(), nn.Dropout(0.1),
            nn.Linear(512, feat_channels), nn.Sigmoid())
        self.pool = nn.AdaptiveAvgPool2d(1)
    def forward(self, feat_map, meta):
        gate = self.gate(meta).unsqueeze(-1).unsqueeze(-1)
        return self.pool(feat_map * gate).flatten(1)

class FusionSkinNet(nn.Module):
    def __init__(self, meta_dim=8):
        super().__init__()
        base = models.resnet50(weights=None)
        self.encoder = nn.Sequential(*list(base.children())[:-2])
        self.lag = LesionAttentionGate(2048, meta_dim)
        self.head = nn.Sequential(
            nn.Linear(2048,512), nn.BatchNorm1d(512), nn.GELU(), nn.Dropout(0.5),
            nn.Linear(512,128),  nn.BatchNorm1d(128), nn.GELU(), nn.Dropout(0.3),
            nn.Linear(128,1))
    def forward(self, img, meta):
        return self.head(self.lag(self.encoder(img), meta))

device = "cuda" if torch.cuda.is_available() else "cpu"
MODEL = FusionSkinNet(meta_dim=8).to(device)
MODEL.load_state_dict(torch.load("best_model.pth", map_location=device))
MODEL.eval()

TRANSFORM = transforms.Compose([
    transforms.Resize((224,224)), transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])

META_COLS = ["age_scaled","sex_female","sex_male",
             "anatom_site_general_anterior torso",
             "anatom_site_general_head/neck",
             "anatom_site_general_lower extremity",
             "anatom_site_general_posterior torso",
             "anatom_site_general_upper extremity"]

@app.get("/", response_class=HTMLResponse)
async def home():
    with open("index.html") as f: return f.read()

@app.post("/predict")
async def predict(
    file: UploadFile = File(...),
    age: float = Form(45.0),
    sex: str   = Form("male"),
    site: str  = Form("anterior torso")
):
    img = Image.open(io.BytesIO(await file.read())).convert("RGB")
    img_t = TRANSFORM(img).unsqueeze(0).to(device)

    # Build meta vector
    meta = [
        (age - 50) / 20,          # age_scaled
        1.0 if sex=="female" else 0.0,
        1.0 if sex=="male"   else 0.0,
        1.0 if site=="anterior torso"    else 0.0,
        1.0 if site=="head/neck"         else 0.0,
        1.0 if site=="lower extremity"   else 0.0,
        1.0 if site=="posterior torso"   else 0.0,
        1.0 if site=="upper extremity"   else 0.0,
    ]
    meta_t = torch.tensor([meta], dtype=torch.float32).to(device)

    with torch.no_grad():
        prob = torch.sigmoid(MODEL(img_t, meta_t)).item()

    risk = "HIGH" if prob > 0.5 else "MODERATE" if prob > 0.2 else "LOW"
    return JSONResponse({"probability": round(prob*100, 2),
                         "risk": risk, "pauc": 0.9686})

if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=8000)
'''

# ── index.html (Frontend) ────────────────────────────────────
html_code = '''<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>SkinCancer AI Detector</title>
<style>
  * { margin:0; padding:0; box-sizing:border-box; }
  body { font-family:'Segoe UI',sans-serif; background:#0D1B2A; color:white; min-height:100vh; }
  .header { background:linear-gradient(135deg,#00C9A7,#4A90E2);
            padding:24px; text-align:center; }
  .header h1 { font-size:2rem; }
  .header p  { opacity:0.85; margin-top:6px; }
  .container { max-width:900px; margin:32px auto; padding:0 20px; }
  .card { background:#162032; border-radius:12px; padding:28px;
          margin-bottom:24px; border:1px solid #1e3a5f; }
  .card h2 { color:#00C9A7; margin-bottom:18px; font-size:1.2rem; }
  .upload-zone { border:2px dashed #4A90E2; border-radius:10px;
                 padding:40px; text-align:center; cursor:pointer;
                 transition:all 0.3s; }
  .upload-zone:hover { background:#1e3a5f; }
  .upload-zone input { display:none; }
  #preview { max-width:100%; max-height:300px; border-radius:8px;
             margin-top:16px; display:none; }
  .form-row { display:grid; grid-template-columns:1fr 1fr 1fr; gap:16px; }
  label { display:block; color:#94A3B8; font-size:0.85rem; margin-bottom:6px; }
  select, input[type=number] {
    width:100%; padding:10px 14px; background:#0D1B2A;
    border:1px solid #334155; border-radius:8px; color:white; font-size:0.95rem; }
  .btn { width:100%; padding:14px; background:linear-gradient(135deg,#00C9A7,#4A90E2);
         border:none; border-radius:10px; color:white; font-size:1.1rem;
         font-weight:bold; cursor:pointer; margin-top:20px; transition:opacity 0.2s; }
  .btn:hover { opacity:0.88; }
  .result { display:none; }
  .gauge { text-align:center; padding:24px; }
  .risk-badge { display:inline-block; padding:8px 28px; border-radius:50px;
                font-size:1.4rem; font-weight:bold; margin:12px 0; }
  .HIGH     { background:#E74C3C33; color:#E74C3C; border:2px solid #E74C3C; }
  .MODERATE { background:#F5A62333; color:#F5A623; border:2px solid #F5A623; }
  .LOW      { background:#00C9A733; color:#00C9A7; border:2px solid #00C9A7; }
  .prob-bar { background:#0D1B2A; border-radius:50px; height:22px;
              margin:16px 0; overflow:hidden; border:1px solid #334155; }
  .prob-fill { height:100%; border-radius:50px;
               background:linear-gradient(90deg,#00C9A7,#E74C3C);
               transition:width 1s ease; }
  .metric-row { display:flex; justify-content:space-around; margin-top:16px; }
  .metric { text-align:center; }
  .metric .val { font-size:1.5rem; font-weight:bold; color:#00C9A7; }
  .metric .key { color:#94A3B8; font-size:0.8rem; margin-top:4px; }
  .loader { display:none; text-align:center; padding:20px; color:#94A3B8; }
  footer { text-align:center; color:#334155; padding:24px; font-size:0.85rem; }
</style>
</head>
<body>
<div class="header">
  <h1>🔬 SkinCancer AI Detector</h1>
  <p>FusionSkinNet — ResNet50 + LesionAttentionGate + LightGBM Hybrid | pAUC: 0.9686</p>
</div>

<div class="container">
  <div class="card">
    <h2>📸 Upload Skin Lesion Image</h2>
    <div class="upload-zone" onclick="document.getElementById(\'file\').click()">
      <div style="font-size:3rem">📁</div>
      <p style="margin-top:8px;color:#94A3B8">Click to upload JPG/PNG image</p>
      <input type="file" id="file" accept="image/*" onchange="previewImg(this)">
      <img id="preview">
    </div>
  </div>

  <div class="card">
    <h2>🏥 Patient Metadata</h2>
    <div class="form-row">
      <div>
        <label>Age</label>
        <input type="number" id="age" value="45" min="1" max="100">
      </div>
      <div>
        <label>Sex</label>
        <select id="sex">
          <option value="male">Male</option>
          <option value="female">Female</option>
        </select>
      </div>
      <div>
        <label>Anatomical Site</label>
        <select id="site">
          <option value="anterior torso">Anterior Torso</option>
          <option value="posterior torso">Posterior Torso</option>
          <option value="upper extremity">Upper Extremity</option>
          <option value="lower extremity">Lower Extremity</option>
          <option value="head/neck">Head / Neck</option>
        </select>
      </div>
    </div>
    <button class="btn" onclick="predict()">🔍 Analyze Lesion</button>
  </div>

  <div class="loader" id="loader">⏳ AI analyzing image + metadata...</div>

  <div class="card result" id="result">
    <h2>📊 Analysis Result</h2>
    <div class="gauge">
      <div class="risk-badge" id="riskBadge">—</div>
      <div style="font-size:2.5rem;font-weight:bold;margin:8px 0" id="probText">0%</div>
      <div style="color:#94A3B8;font-size:0.9rem">Malignancy Probability</div>
      <div class="prob-bar">
        <div class="prob-fill" id="probFill" style="width:0%"></div>
      </div>
      <div class="metric-row">
        <div class="metric"><div class="val">0.9686</div><div class="key">Model pAUC</div></div>
        <div class="metric"><div class="val" id="probVal">—</div><div class="key">This Image</div></div>
        <div class="metric"><div class="val">0.9790</div><div class="key">Model AUC</div></div>
      </div>
    </div>
    <p style="color:#94A3B8;font-size:0.8rem;text-align:center;margin-top:12px">
      ⚠️ For research purposes only. Not a substitute for medical diagnosis.
    </p>
  </div>
</div>
<footer>FusionSkinNet Phase 3 — ISIC 2024 | pAUC 0.9686</footer>

<script>
function previewImg(input) {
  if (input.files && input.files[0]) {
    const reader = new FileReader();
    reader.onload = e => {
      const img = document.getElementById("preview");
      img.src = e.target.result;
      img.style.display = "block";
    };
    reader.readAsDataURL(input.files[0]);
  }
}

async function predict() {
  const file = document.getElementById("file").files[0];
  if (!file) { alert("Please upload an image first!"); return; }

  document.getElementById("loader").style.display = "block";
  document.getElementById("result").style.display = "none";

  const form = new FormData();
  form.append("file", file);
  form.append("age",  document.getElementById("age").value);
  form.append("sex",  document.getElementById("sex").value);
  form.append("site", document.getElementById("site").value);

  const res  = await fetch("/predict", { method:"POST", body:form });
  const data = await res.json();

  document.getElementById("loader").style.display    = "none";
  document.getElementById("result").style.display    = "block";
  document.getElementById("probText").textContent    = data.probability + "%";
  document.getElementById("probVal").textContent     = data.probability + "%";
  document.getElementById("probFill").style.width    = data.probability + "%";

  const badge = document.getElementById("riskBadge");
  badge.textContent  = data.risk + " RISK";
  badge.className    = "risk-badge " + data.risk;
}
</script>
</body>
</html>
'''

# ── Save files ───────────────────────────────────────────────
os.makedirs('/kaggle/working/webapp', exist_ok=True)
with open('/kaggle/working/webapp/app.py', 'w')      as f: f.write(app_code)
with open('/kaggle/working/webapp/index.html', 'w')  as f: f.write(html_code)

print("✅ webapp/app.py     saved")
print("✅ webapp/index.html saved")
print("""
──────────────────────────────────────────────────────
Run karne ke liye (local machine pe):
  cd webapp
  pip install fastapi uvicorn python-multipart pillow torch torchvision
  cp /kaggle/working/best_model.pth .
  python app.py
  → Open: http://localhost:8000
──────────────────────────────────────────────────────
""")

✅ webapp/app.py     saved
✅ webapp/index.html saved

──────────────────────────────────────────────────────
Run karne ke liye (local machine pe):
  cd webapp
  pip install fastapi uvicorn python-multipart pillow torch torchvision
  cp /kaggle/working/best_model.pth .
  python app.py
  → Open: http://localhost:8000
──────────────────────────────────────────────────────



In [ ]:
# ZIP banao download ke liye
import shutil
shutil.copy('/kaggle/working/best_model.pth', '/kaggle/working/webapp/best_model.pth')
shutil.make_archive('/kaggle/working/webapp_download', 'zip', '/kaggle/working/webapp')
print("✅ webapp_download.zip ready — right panel se download karo")

✅ webapp_download.zip ready — right panel se download karo
